# C1 · 為什麼要「非中心化參數化」？—— funnel 的故事

> 階層模型有個惡名昭彰的陷阱：當群組層級的標準差 $\tau$（或 $\sigma_a$）很小時，
> 後驗會擠成一個**漏斗（funnel）**，MCMC 在漏斗頸部採不動 → 出現 divergences。
> 解法是**重參數化**（計劃書 §9 / 主題五第三卡）。用經典的 Eight Schools 把它演出來。

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
import data, models
y, sigma, names = data.eight_schools()
id_c  = models.fit_eight_schools(y, sigma, centered=True)
id_nc = models.fit_eight_schools(y, sigma, centered=False)
print('中心化   divergences =', int(id_c.sample_stats['diverging'].sum()))
print('非中心化 divergences =', int(id_nc.sample_stats['diverging'].sum()))

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [mu, tau, theta]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1 seconds.


There were 113 divergences after tuning. Increase `target_accept` or reparameterize.


...

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [mu, tau, theta_raw]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1 seconds.


中心化   divergences = 113
非中心化 divergences = 0


## 1 · 漏斗現形

![funnel](../figures/05_funnel.png)

**左（中心化）**：採樣器下不到漏斗頸部（低 $\log\tau$），約 **113 個 divergences**（紅點）全擠在頸口。
**右（非中心化）**：一路探到 $\log\tau \approx -6$，**0 divergences**。

## 2 · 為什麼非中心化有效

- **中心化**：$a_j \sim \mathcal{N}(\mu_a, \sigma_a)$。當 $\sigma_a$ 小，$a_j$ 的可行範圍被 $\sigma_a$ 綁死——
  參數之間強烈相依，形成漏斗，NUTS 的步長無法同時適配寬肚與窄頸。
- **非中心化**：$a_j = \mu_a + \sigma_a \cdot \tilde a_j$，其中 $\tilde a_j \sim \mathcal{N}(0,1)$。
  現在 $\tilde a_j$ 的幾何**不再依賴** $\sigma_a$，漏斗被「拉直」，採樣器暢行無阻。

```python
# 中心化（容易 diverge）
a = pm.Normal('a', mu_a, sigma_a, shape=J)
# 非中心化（建議）
a = pm.Deterministic('a', mu_a + sigma_a * pm.Normal('a_raw', 0, 1, shape=J))
```
> 這個 $x = \mu + \sigma\cdot\varepsilon$ 的技巧，就是變分推論裡的**重參數化**——在 MCMC 裡它同樣救命。

## 3 · 這和 Radon 的收縮有什麼關係？

同一套階層機制。差別在**資料量**：
- **Eight Schools**：只有 8 組、每組一個帶大誤差的觀測 → $\tau$ 難識別、常接近 0 → 漏斗嚴重。
- **Radon**：919 戶、85 郡 → $\sigma_a$ 識別良好（≈0.32，見 NB1）→ 漏斗不明顯，中心化其實也採得動。

所以：**資料少時務必用非中心化**；資料多時兩者皆可，但非中心化是安全的預設。

## 重點

> 階層模型的 funnel 是採樣幾何問題，不是模型錯。**非中心化重參數化**把漏斗拉直，
> 是寫任何階層貝葉斯模型都該有的反射動作——尤其群組數少、每組資料少時。